In [ ]:
import os

print("=== YOUR UPLOAD (lcam-v5-finetune) ===")
for root, dirs, files in os.walk('/kaggle/input/lcam-v5-finetune'):
    if root.count('/') <= 5:
        print(root)
        if files:
            print("   files:", files[:5])

print("\n=== D-FIRE (smoke-fire-detection-yolo) ===")
for root, dirs, files in os.walk('/kaggle/input/smoke-fire-detection-yolo'):
    if root.count('/') <= 5:
        print(root)
        if dirs:
            print("   subdirs:", dirs[:6])

In [ ]:
import os
print("What's in /kaggle/input:")
print(os.listdir('/kaggle/input'))

In [ ]:
import os
print("Inside /kaggle/input/datasets:")
for root, dirs, files in os.walk('/kaggle/input/datasets'):
    if root.count('/') <= 6:
        print(root)
        if files:
            print("   files:", files[:5])

In [ ]:
import os
# Full walk to find YOUR upload + the D-Fire train/valid structure
for root, dirs, files in os.walk('/kaggle/input/datasets'):
    if root.count('/') <= 8:
        print(root)
        if files:
            print("   files:", files[:4])

In [ ]:
import shutil, os

BASE = '/kaggle/input/datasets/punnamrahul/lcam-v5-finetune/kaggle_v5_upload'

# Clean any previous copy
for p in ['/kaggle/working/YOLOX', '/kaggle/working/finetune_lcam_v5.py', '/kaggle/working/lcam_yolox_wildfire_v32_compat.pth']:
    if os.path.exists(p):
        shutil.rmtree(p) if os.path.isdir(p) else os.remove(p)

# Copy YOLOX, script, weights to working dir (writable)
shutil.copytree(f'{BASE}/YOLOX', '/kaggle/working/YOLOX')
shutil.copy(f'{BASE}/finetune_lcam_v5.py', '/kaggle/working/')
shutil.copy(f'{BASE}/lcam_yolox_wildfire_v32_compat.pth', '/kaggle/working/')

# Verify v5 is present
lcam = '/kaggle/working/YOLOX/yolox/models/lcam.py'
with open(lcam) as f:
    src = f.read()
print("v5 check -> Hardsigmoid:", 'Hardsigmoid' in src, "| StagedGlobalAvgPool:", 'StagedGlobalAvgPool' in src)

# Install deps
os.system('pip install loguru thop tabulate -q')
print("Setup done. Working dir:", os.listdir('/kaggle/working'))

In [ ]:
# Fix dataset path and val/valid split name in the script
script = '/kaggle/working/finetune_lcam_v5.py'
with open(script) as f:
    code = f.read()

# Correct dataset root
code = code.replace(
    "'/kaggle/input/smoke-fire-detection-yolo/data'",
    "'/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data'"
)
# The val split is named 'val' not 'valid'
code = code.replace("split='valid'", "split='val'")

with open(script, 'w') as f:
    f.write(code)
print("Patched dataset path and val split name.")
# confirm
print("dataset root in script:", "sayedgamal99" in code)
print("val split fixed:", "split='val'" in code)

In [ ]:
!cd /kaggle/working && python finetune_lcam_v5.py